# Q1 Time-Dependent Recycling-Kernel Validation

This notebook validates the fixed-basis Lindblad-safe compact interpolation for time-dependent electric fields. The candidate uses the production patch grid and full recycling-kernel dissipator. The reference precomputes direct aligned compact bundles on a denser field grid and interpolates those Liouvillians during propagation.

In [ ]:
from __future__ import annotations

import importlib.util
import pathlib
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def load_validation_module():
    cwd = pathlib.Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        script_path = (
            candidate
            / "examples"
            / "effective hamiltonian"
            / "validate_q1_time_dependent_recycling_kernel.py"
        )
        if script_path.exists():
            spec = importlib.util.spec_from_file_location(
                "validate_q1_time_dependent_recycling_kernel",
                script_path,
            )
            module = importlib.util.module_from_spec(spec)
            sys.modules[spec.name] = module
            spec.loader.exec_module(module)
            return module
    raise FileNotFoundError("Could not locate validation script")


plt.rcParams.update({"font.size": 14})
v = load_validation_module()
ehr = v.load_runtime_module()
print("validation script:", pathlib.Path(v.__file__).resolve())

## Configuration

`REFERENCE_POINTS` controls the dense direct-compact reference grid. Increasing it makes setup slower but reduces reference interpolation error.

In [ ]:
REFERENCE_POINTS = 41
REFERENCE_FIELDS_VCM = np.linspace(0.0, 50.0, REFERENCE_POINTS)

cases = [
    (
        "linear_ramp_0_to_50_vcm",
        lambda t: 50.0 * float(t) / v.T_FINAL,
    ),
    (
        "sinusoid_25_pm_20_vcm",
        lambda t: 25.0 + 20.0 * np.sin(2.0 * np.pi * float(t) / v.T_FINAL),
    ),
]

print(
    {
        "reference_points": REFERENCE_POINTS,
        "t_final_us": 1e6 * v.T_FINAL,
        "rabi_rate_MHz": v.RABI_RATE / (2.0 * np.pi * 1e6),
    }
)

## Build Candidate And Reference

The reference setup is the expensive step because it builds direct compact systems at every dense reference field.

In [ ]:
setup_start = time.perf_counter()
model = v.build_model(ehr)
candidate_data = v.precompute_candidate(ehr, model)
candidate_setup_s = time.perf_counter() - setup_start

reference_start = time.perf_counter()
reference_data = v.precompute_reference(ehr, model, REFERENCE_FIELDS_VCM)
reference_setup_s = time.perf_counter() - reference_start

rho0 = v.uniform_density(model.n_effective_states, np.asarray(model.ground_indices))
excited_indices = np.asarray(model.excited_indices, dtype=np.int64)

pd.DataFrame(
    [
        {
            "candidate_setup_s": candidate_setup_s,
            "reference_setup_s": reference_setup_s,
            "n_effective_states": model.n_effective_states,
            "candidate_fields": len(candidate_data["fields"]),
            "reference_fields": len(reference_data["fields"]),
        }
    ]
)

## Run Time-Dependent Comparisons

In [ ]:
results = []
for name, field_at in cases:
    results.append(
        v.compare_case(
            name,
            field_at,
            candidate_data,
            reference_data,
            rho0,
            excited_indices,
        )
    )

summary_rows = []
for result in results:
    row = {"case": result["case"]}
    row.update(result["errors"])
    row.update(result["solver"])
    row["candidate_photons"] = result["candidate"]["photons"]
    row["reference_photons"] = result["reference"]["photons"]
    row["candidate_final_excited"] = result["candidate"]["final_excited_population"]
    row["reference_final_excited"] = result["reference"]["final_excited_population"]
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df

## Error Summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)

x = np.arange(len(summary_df))
axes[0].bar(x - 0.2, np.abs(summary_df["photons_rel"]), width=0.4, label="photons")
axes[0].bar(
    x + 0.2,
    np.abs(summary_df["excited_integral_rel"]),
    width=0.4,
    label="excited integral",
)
axes[0].set_xticks(x, summary_df["case"], rotation=20, ha="right")
axes[0].set_ylabel("absolute relative error")
axes[0].set_title("Integrated observables")
axes[0].grid(axis="y", alpha=0.25)
axes[0].legend()

axes[1].bar(x - 0.2, np.abs(summary_df["final_excited_rel"]), width=0.4, label="final")
axes[1].bar(x + 0.2, np.abs(summary_df["max_excited_rel"]), width=0.4, label="max trace")
axes[1].set_xticks(x, summary_df["case"], rotation=20, ha="right")
axes[1].set_ylabel("absolute relative error")
axes[1].set_title("Excited-state population")
axes[1].grid(axis="y", alpha=0.25)
axes[1].legend()

plt.show()

## Interpretation

- Integrated photon and excited-population errors are the primary validation metrics.
- Final excited-state relative error can look larger because the final excited population is small.
- If errors need to be reduced further, increase the production patch density or add targeted patch points where the time-dependent trajectory spends time.